# Which prompt change is real, and how do you prove it before and after release?

Question: on six office tasks with exact answers, which prompt technique really raises the success rate of a small model, does an automatic prompt optimizer keep its gain on new items, and how do you prove a change with an offline experiment and an online A/B test?

Setup: a CPU notebook. It uses the Python standard library, numpy, pandas, scipy and matplotlib. It calls the NVIDIA-hosted, OpenAI-compatible API (`https://integrate.api.nvidia.com/v1`) and needs an `NVIDIA_API_KEY` secret. Parts A to C call the API: at most 1,500 calls, 4 in flight. Parts D to F need no API. They run on the measured rates of parts A to C. If the API did not run, they print `FALLBACK` and use stated rates. The small model is `nvidia/nemotron-3.5-bolt-e4b`. The mid model is `nvidia/nemotron-3.5-lightning-30b-a3b`. The large model `nvidia/nemotron-3-ultra-550b-a55b` only writes prompts in part C. Reasoning is off. Requests are paced: bolt has no gap, lightning and ultra wait 1.7 s between requests, because the API answers 429 above about 40 requests a minute for them (measured on 2026-11-25). Lightning and ultra get under 200 calls in all (90 zero-shot and 90 chain-of-thought on lightning, 4 on ultra). The bulk (repeats, few-shot sweeps, optimization) runs on bolt. If time gets short, cut the lightning chain-of-thought run first. All items are fictional. Expected run time: 10 to 20 minutes.

Terms used below:

- **Use case**: one kind of office task. This notebook has six: email triage, action items from meeting notes, document question answering, summary with given facts, question to SQL, and error diagnosis. Each has 15 items and one exact answer that code can check.
- **Exact grader**: plain code that says right, wrong or unreadable. It never asks a model.
- **Prompt engineering**: you change the input text. The model weights stay the same (`selecting-large-language-model-customization-techniques.txt`).
- **System prompt**: standing rules that come before the task.
- **Zero-shot**: the task with no example. **Few-shot**: you add a few example pairs of task and answer (`large-language-models.txt`, `selecting-large-language-model-customization-techniques.txt`).
- **Chain of thought**: you ask the model to think step by step before the final answer (`an-easy-introduction-to-llm-reasoning-ai-agents-and-test-time-scaling.txt`).
- **Structured output**: the reply must fit a JSON schema. `guided-decoding.txt` says constrained decoding makes a reply fit a grammar such as a JSON schema. This notebook sends `response_format` and counts the replies that still fail.
- **Prompt optimization**: a loop. Errors go to a larger model. It writes the next prompt. Each prompt is scored on dev items, then on held-out items.
- **Offline experiment**: two prompts run on the same fixed set of items (a golden set), before any user sees the change. **A/B test**: users are split at random between old and new, and you compare what happens. `announcing-nvidia-merlin-application-framework-for-deep-recommender-systems.txt` says a model is first evaluated offline, then online, typically through A/B testing.
- **Dev items and test items**: you choose a setting on dev items. You measure it on test items that the choice never saw.
- **Wilson interval**: the 95% interval of a success rate. **Exact McNemar test**: a paired test. It counts only the items where the two systems differ.
- **Power**: the chance that a test finds a gain that is really there. **MDE**: the smallest gain worth shipping.
- **Peeking**: you look at a running test many times and stop at the first small p-value. **Sample ratio mismatch (SRM)**: the real split of users is not the planned split. It means a bug.
- **Novelty effect**: users react to what is new. The lift fades. **Guardrail metric**: a metric that must not get worse (latency, cost), even when the main metric improves.
- **Simpson's paradox**: a total goes one way and every segment goes the other way, because the segments are mixed in different shares.
- **Shadow**: the new version runs on live traffic, but users see only the old answer. **Canary**: the new version serves a small share of users, with automatic rollback.

## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Question | What you measure | Your guess |
|---|---|---|
| Which use case does the small model fail most at zero-shot? | Success per use case against the baseline | |
| Which technique helps most, and which one hurts? | Success against zero-shot, exact McNemar p-value | |
| Do more few-shot examples always help? | `few_1`, `few_3`, `few_5` | |
| Does structured output remove format errors? Does it raise success? | `format_error` column and success | |
| Does the same prompt give the same result twice at temperature 0? | Items whose result changed between two runs | |
| Does the technique that wins on dev also win on test? | Dev and test success of the pick | |
| Does the optimized prompt keep its gain on held-out items? | Dev gain against test gain, per round | |
| How many items does a golden set need to show a 5-point gain? | Power table of part D | |
| How often does an A/A test show a false win with 10 looks? | Part E2 | |
| Which guardrail stops a win that is too slow or too costly? | Part E5 | |
| Does the ranking of use cases change when a failure costs repair time? | Part F | |

In [ ]:
# 1. Setup: constants, API key, the part runner, one chat helper with timeout, retry, call cap and early stop
import http.client, json, math, os, random, re, sqlite3, threading, time, traceback, urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor
from datetime import date, timedelta
import numpy as np, pandas as pd
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE = os.environ.get("NIM_BASE_URL", "https://integrate.api.nvidia.com/v1")
SMALL, MID, BIG = "nvidia/nemotron-3.5-bolt-e4b", "nvidia/nemotron-3.5-lightning-30b-a3b", "nvidia/nemotron-3-ultra-550b-a55b"
SEED, CONCURRENCY, MAX_CALLS, ABORT_AFTER = 42, 4, 1500, 5
REQ_TIMEOUT_S, MAX_RETRIES, BACKOFF_S, RETRY_CAP_S = 60, 4, 2.0, 30.0
TOTAL_BUDGET_S = 50 * 60
PACE_S = 1.7   # measured 2026-11-25: bolt took 551+ requests a minute; lightning and ultra answer 429 above about 40 a minute (no Retry-After)
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
T0 = time.time()

def get_key():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("NVIDIA_API_KEY")
    except Exception:
        return os.environ.get("NVIDIA_API_KEY", "")

KEY = get_key()
print("key found:", bool(KEY))   # never print the key itself

def scrub(s):
    return str(s).replace(KEY, "***") if KEY else str(s)

STATUS, DEADLINE, CALL_LIMIT = [], T0 + TOTAL_BUDGET_S, MAX_CALLS
def part(name, fn, budget_s=600):   # a batch run stops at the first exception, so every part runs through here
    global DEADLINE
    t = time.time(); DEADLINE = min(t + budget_s, T0 + TOTAL_BUDGET_S); ok, note = True, ""
    try:
        fn()
    except Exception:
        ok, tb = False, scrub(traceback.format_exc()); note = tb.strip().splitlines()[-1][:120]; print(tb[-1500:])
    STATUS.append(dict(part=name, ok=ok, seconds=round(time.time() - t), budget_s=budget_s, over_budget=time.time() - t > budget_s, note=note))

LOCK, CALLS = threading.Lock(), dict(n=0, ok=0, fail=0)
NEXT, CODES = {}, {}   # next allowed send time per model; counts of failed attempts per (model, status)
def pace(model, deadline):   # one slot per PACE_S for every model except bolt; the wait never goes past the deadline
    if "bolt" in model: return True
    with LOCK:
        t = max(time.time(), NEXT.get(model, 0)); ok = deadline is None or t < deadline
        if ok: NEXT[model] = t + PACE_S
    if ok: time.sleep(max(0, t - time.time()))
    return ok
def short_name(m): return m.split("/")[-1]
def chat(model, messages, extra=None, max_tokens=500, deadline=None):
    out = dict(ok=False, error="", content="", finish="", ptok=0, ctok=0, latency_s=float("nan"))
    with LOCK:
        if not KEY: out["error"] = "no key"
        elif CALLS["n"] >= CALL_LIMIT: out["error"] = "cap"
        elif CALLS["ok"] == 0 and CALLS["fail"] >= ABORT_AFTER: out["error"] = "aborted"   # the first 5 calls all failed
    if out["error"]:
        return out
    body = {"model": model, "messages": messages, "max_tokens": max_tokens, "temperature": 0, "seed": SEED,
            "chat_template_kwargs": {"enable_thinking": False}, **(extra or {})}
    data = json.dumps(body).encode()
    for attempt in range(MAX_RETRIES + 1):
        if not pace(model, deadline): out["error"] = "deadline"; break
        with LOCK:
            if CALLS["n"] >= CALL_LIMIT: out["error"] = "cap"; return out
            CALLS["n"] += 1
        timeout = REQ_TIMEOUT_S if deadline is None else max(1, min(REQ_TIMEOUT_S, deadline - time.time()))
        req = urllib.request.Request(BASE + "/chat/completions", data, {"Authorization": "Bearer " + KEY, "Content-Type": "application/json"})
        t, wait = time.time(), None
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                j = json.loads(r.read())
            ch = j["choices"][0]; u = j.get("usage") or {}
            out.update(ok=True, error="", content=ch["message"].get("content") or "", finish=ch.get("finish_reason") or "",
                       ptok=u.get("prompt_tokens") or 0, ctok=u.get("completion_tokens") or 0, latency_s=time.time() - t)
            with LOCK: CALLS["ok"] += 1
            return out
        except urllib.error.HTTPError as e:   # 429 and 5xx are retried, other codes are final
            out["error"] = f"HTTP {e.code}"
            if e.code != 429 and e.code < 500:
                with LOCK: CODES[(short_name(model), out["error"])] = CODES.get((short_name(model), out["error"]), 0) + 1
                break
            try: wait = float(e.headers.get("Retry-After", ""))
            except ValueError: wait = None
        except (ValueError, KeyError, IndexError, TypeError, AttributeError) as e:   # malformed JSON or missing fields
            out["error"] = "bad reply " + type(e).__name__
            with LOCK: CODES[(short_name(model), out["error"])] = CODES.get((short_name(model), out["error"]), 0) + 1
            break
        except (OSError, http.client.HTTPException) as e:   # timeout, refused, reset, cut-off body
            out["error"] = type(getattr(e, "reason", e)).__name__
        with LOCK: CODES[(short_name(model), out["error"])] = CODES.get((short_name(model), out["error"]), 0) + 1
        if attempt == MAX_RETRIES: break
        wait = min(BACKOFF_S * 2 ** attempt if wait is None else wait, RETRY_CAP_S)
        if deadline is not None and time.time() + wait >= deadline: break
        time.sleep(wait)
    with LOCK: CALLS["fail"] += 1
    return out

def pmap(fn, items, deadline):   # at most CONCURRENCY calls at once; items not started by the deadline give None
    with ThreadPoolExecutor(CONCURRENCY) as ex:
        return list(ex.map(lambda x: fn(x) if time.time() < deadline else None, items))

def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - h) / d, (c + h) / d)

def mcnemar(a, b):   # exact McNemar on paired 0/1 arrays: (only a right, only b right, two-sided p)
    a, b = np.asarray(a), np.asarray(b); x, y = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
    return x, y, (1.0 if x + y == 0 else stats.binomtest(x, x + y, 0.5).pvalue)

assert abs(wilson(5, 10)[0] + wilson(5, 10)[1] - 1) < 1e-9 and abs(wilson(0, 10)[0]) < 1e-12
assert mcnemar([1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0] * 10)[2] < 0.01 and mcnemar([1, 0], [0, 1])[2] == 1.0

## Part A. Six productivity use cases, exact graders

Question: how well does a model do six office tasks with no help, and how far is that from a baseline that always gives the same answer?

The exam asks for productivity use cases of LLM-powered applications. The six here are our own examples, not a list from a source file. Each item is built so that the small model can fail in a natural way: near-miss labels with a stated precedence rule, look-alike names, a replaced fact, a SQL query that needs a join, HAVING or NOT EXISTS, and a loud error that is only a symptom. Cells 2 and 3 test every grader on right replies, fenced replies, wrong replies and garbage. Cut, empty, unreadable and failed replies are outcomes of their own. They are never dropped.

In [ ]:
# 2. Part A data: 6 use cases x 15 fictional items, one exact answer each. Hand-written or built from fixed seeds. No API.
# Each item: id, uc, idx, body (input), fmt (output rule), rv (the right answer as a Python value), gold (what the grader needs).
TRIAGE_LABELS = ["billing_error", "service_outage", "account_access", "refund_request", "invoice_request", "sales_inquiry"]
CAUSES = ["disk_full", "expired_certificate", "dns_failure", "connection_pool_exhausted", "null_reference", "out_of_memory",
          "permission_denied", "missing_config", "schema_mismatch", "timezone_bug", "race_condition", "rate_limit"]
UCS = ["triage", "actions", "docqa", "summary", "sql", "diagnosis"]
FIELD = dict(triage="label", actions="actions", docqa="answer", summary="summary", sql="sql", diagnosis="cause")
DOW = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# ---- 1. email triage (hand-written; near-miss labels and a stated precedence rule)
TRIAGE_RULES = ("Labels: billing_error (the amount, quantity or date of a charge or invoice is wrong), service_outage (the product or API is down or failing for many users), "
                "account_access (login, password, 2FA or permissions for one user), refund_request (money back for a charge that was correct), "
                "invoice_request (a copy or detail of an invoice, no money problem), sales_inquiry (price, quote, demo or contract).\n"
                "If an email fits two labels, use the one that comes first in this order: billing_error, service_outage, account_access, refund_request, invoice_request, sales_inquiry.")
TRIAGE_FMT = "Reply with the label only."
TRIAGE = [
 ("My card was charged $240 on 3 Nov but my plan costs $120. Please send the difference back.", "billing_error"),
 ("I returned the two monitors last week, tracking attached. Please refund the $310 to my card.", "refund_request"),
 ("Our finance team needs a PDF copy of invoice INV-2291 with our VAT number printed on it.", "invoice_request"),
 ("None of our 40 agents can reach the dashboard since 09:10. It just spins. Is something down?", "service_outage"),
 ("I changed phones and the 2FA code no longer arrives. I cannot log in to my own account.", "account_access"),
 ("We have 200 seats and compare you with another vendor. Can you send a quote for the Enterprise tier and book a demo?", "sales_inquiry"),
 ("I cancelled on 1 Oct, but you billed me again on 1 Nov. Cancel it for good and give that money back.", "billing_error"),
 ("The app is down for everyone in our office, and I also cannot reset my password. Please help.", "service_outage"),
 ("I no longer need the Pro plan. The $49 charge was correct, I just forgot to cancel. Please refund this month.", "refund_request"),
 ("Can you resend the invoice for October? Also, what would 50 extra seats cost?", "invoice_request"),
 ("I typed my password wrong five times and now I am locked out. The site works fine for my colleagues.", "account_access"),
 ("Invoice INV-3310 lists 12 licences, but we bought 10. Please correct it and send a new PDF.", "billing_error"),
 ("Is your API returning 503 for anyone else? Our checkout has failed for 20 minutes.", "service_outage"),
 ("Thanks for the demo last week. We liked it. Please send the contract and the price for 3 years.", "sales_inquiry"),
 ("Please give my new colleague Ana access to the shared project folder. She cannot open it.", "account_access"),
]
TRIAGE_EX = [
 ("You charged my card twice for the same order today. Please fix it.", "billing_error"),
 ("The whole site shows a blank page for every customer since noon.", "service_outage"),
 ("Please change the email on my login. I can no longer open my old mailbox.", "account_access"),
 ("I changed my mind about the headset I bought. It is unused. Can I send it back for my money?", "refund_request"),
 ("Do you have a rough price for 15 seats on the Team tier?", "sales_inquiry"),
]

# ---- 2. meeting notes to action items (built from a seed; dates computed with datetime)
NAMES = ["Dana Ortiz", "Dana Ortega", "Marcus Webb", "Marcus Webber", "Ilse Brandt", "Ilse Brandt-Koh", "Tomas Reyes", "Tomas Reyna", "Priya Nair", "Priya Nayar", "Lena Fischer", "Yusuf Demir"]
WORK = ["send the revised budget", "book the vendor call", "update the risk register", "share the test plan", "approve the draft contract",
        "collect the survey results", "fix the onboarding checklist", "publish the release notes", "order the demo hardware", "review the security findings"]
ACT_FMT = ('Reply with JSON only: {"actions": [{"owner": "<full name>", "due": "YYYY-MM-DD"}]}. One entry for each task that a named person agreed to do. '
           'Skip ideas that nobody took, tasks that are already done and tasks that were dropped.')
def ordinal(n): return f"{n}{'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')}"
def due_phrase(r, d0):
    k = r.choice(["weekday", "days", "dom", "tomorrow", "month_end", "weeks"])
    if k == "weekday": w = r.randrange(7); return f"by {DOW[w]}", d0 + timedelta((w - d0.weekday()) % 7 or 7)
    if k == "days": n = r.randint(2, 9); return f"in {n} days", d0 + timedelta(n)
    if k == "dom": n = r.randint(d0.day + 1, 28); return f"on the {ordinal(n)}", d0.replace(day=n)
    if k == "tomorrow": return "tomorrow", d0 + timedelta(1)
    if k == "weeks": return "in two weeks", d0 + timedelta(14)
    return "by the end of the month", (d0.replace(day=28) + timedelta(4)).replace(day=1) - timedelta(1)
def make_actions(seed):
    r = random.Random(seed); d0 = date(2026, r.randint(1, 12), r.randint(1, 20))
    pair = r.randrange(5) * 2; owners = [NAMES[pair], NAMES[pair + 1], r.choice([n for i, n in enumerate(NAMES) if i // 2 != pair // 2])]
    works = r.sample(WORK, 6); acts, lines = [], []
    for o, w in zip(owners, works):
        ph, d = due_phrase(r, d0); acts.append((o, d.isoformat()))
        lines.append(r.choice([f"{o} will {w} {ph}.", f"Action for {o}: {w}, due {ph}.", f"{o} takes it to {w} {ph}."]))
    if len({a for a in acts}) < 3: return make_actions(seed + 1000)
    p = r.choice(NAMES)
    lines += [f"{p} suggested that we {works[3]}, but nobody took it.", f"{r.choice(owners)} already did the job to {works[4]} on Monday.", f"The plan to {works[5]} was dropped."]
    r.shuffle(lines)
    body = (f"Meeting date: {d0.isoformat()} ({DOW[d0.weekday()]}).\nDue dates: a weekday name means the first such day after the meeting date. "
            f"'In N days' counts from the meeting date. 'The end of the month' is the last day of the meeting month.\n\nNotes:\n" + "\n".join(lines))
    return body, [dict(owner=o, due=d) for o, d in acts]
ACTIONS = [make_actions(100 + i) for i in range(15)]
ACTIONS_EX = [make_actions(900 + i) for i in range(5)]

# ---- 3. document question answering (five templates, three instances each; distractor facts that look alike)
def dfmt(d): return f"{d.day} {d:%B}"
def q_amend(r):
    item = r.choice(["meal allowance per day", "parking allowance per month", "laptop budget per hire"]); old = r.randrange(30, 80, 5); new = old + r.choice([10, 15, 20, 25])
    eff = date(2026, r.randint(6, 9), 1); amend = eff - timedelta(r.randint(20, 40)); ask = r.choice([eff - timedelta(r.randint(3, 20)), eff + timedelta(r.randint(2, 25))])
    doc = (f"Policy memo. The {item} is ${old}. An amendment signed on {amend:%d %B %Y} raises the {item} to ${new}, effective {eff:%d %B %Y}. "
           f"A separate amendment of {amend - timedelta(9):%d %B %Y} cut the guest allowance to ${old - 5}. The visitor badge fee stays at ${new + 5}.")
    return doc, f"What is the {item} on {ask:%d %B %Y}? Answer with the number only.", float(new if ask >= eff else old), "num"
def q_vendor(r):
    b = r.choice(["Brightline", "Northgate", "Clearwater", "Stonebridge"]); v = [f"{b} Ltd", f"{b} Group", f"{b}s Ltd"]; days = r.sample(DOW[:5], 3); fee = r.sample(range(40, 95, 5), 3)
    doc = (" ".join(f"{v[i]} delivers on {days[i]}s and charges ${fee[i]} per pallet." for i in range(3)) +
           f" Before the 2025 contract, {v[0]} charged ${fee[1] + 5} per pallet.")
    i = r.randrange(3)
    return doc, (f"What does {v[i]} charge per pallet now?" if r.random() < 0.5 else f"What does the vendor that delivers on {days[i]}s charge per pallet now?") + " Answer with the number only.", float(fee[i]), "num"
def q_total(r):
    qty, price, thr, pct, ship = r.randint(15, 40), r.choice([25, 35, 45, 60]), r.choice([20, 25]), r.choice([10, 15, 20]), r.choice([18, 24, 30])
    doc = (f"Quote Q-{r.randint(100, 999)}. Licences cost ${price} each. A {pct}% discount applies to the licences when the order has {thr} or more of them. "
           f"Shipping is a flat ${ship} and is not discounted. The Basic tier costs ${price - 10} each, but this quote is for the Pro tier. The customer orders {qty} licences.")
    tot = qty * price * (100 - (pct if qty >= thr else 0)) / 100 + ship
    return doc, "What is the total cost in dollars? Answer with the number only.", float(tot), "num"
def q_remote(r):
    a, b, c, n = r.choice([3, 4]), r.choice([1, 2]), r.choice([1, 2]), r.choice([60, 90]); who = r.choice(["team lead", "analyst"]); k = r.choice([n - 40, n + 30]); freeze = r.random() < 0.5
    doc = (f"Staff may work remotely up to {a} days per week. Staff in their first {n} days may work remotely only {b} days per week. Team leads are exempt from the first-{n}-days limit. "
           f"In a release freeze week nobody may work remotely more than {c} days. The office is closed for visitors on Fridays.")
    cap = a if (who == "team lead" or k > n) else b
    return doc, (f"Sam Hale is a {who} who joined {k} days ago. This is {'a release freeze week' if freeze else 'a normal week'}. "
                 "How many days may Sam work remotely? Answer with the number only."), float(min(cap, c) if freeze else cap), "num"
def q_sla(r):
    d = date(2026, r.randint(1, 11), r.randint(1, 20)); n = r.choice([2, 3]); hol = d + timedelta(r.randint(1, 3))
    doc = (f"Ticket T-{r.randint(1000, 9999)} was opened on {d:%A %d %B %Y}. The first reply is due within {n} business days. Saturdays and Sundays are not business days. "
           f"{hol:%d %B %Y} is a public holiday and is not a business day. Resolution is due within 10 calendar days of opening.")
    x, left = d, n
    while left:
        x += timedelta(1)
        if x.weekday() < 5 and x != hol: left -= 1
    return doc, "On which date is the first reply due? Answer in the form YYYY-MM-DD.", x.isoformat(), "date"
def make_docqa(seed):
    r = random.Random(seed); doc, q, ans, kind = [q_amend, q_vendor, q_total, q_remote, q_sla][seed % 5](r)
    return f"Document:\n{doc}\n\nQuestion: {q}", (ans, kind)
DOCQA = [make_docqa(200 + i) for i in range(15)]
DOCQA_EX = [make_docqa(800 + i) for i in range(5)]

# ---- 4. summary that must contain given facts (and must not repeat the stale ones)
def has(text, fact): return re.search(r"(?<![\w.$])" + re.escape(fact).replace(r"\ ", r"\s+") + r"(?![\w])", text, re.I) is not None
def make_summary(seed):
    r = random.Random(seed); k = seed % 3; n, est = r.randint(60, 300), r.randint(60, 300) + 400; names = [x for x in NAMES]
    if k == 0:
        i, who, build, t1, t2 = f"INC-{r.randint(1000, 9999)}", r.choice(names), f"b{r.randint(400, 899)}", f"{r.randint(9, 11)}:{r.randrange(0, 60, 5):02d}", f"{r.randint(13, 16)}:{r.randrange(0, 60, 5):02d}"
        text = (f"Status note. Ticket {i} was opened at {t1} after the payment queue stopped. The first estimate said {est} customers were hit. After the log review the final count is {n} customers. "
                f"{who} led the response and rolled back build {build}. An early post blamed a full disk, but the final review found an expired token. The queue was normal again at {t2}. The next review is on Friday.")
        req, forb = [i, str(n), build, t2], [str(est), "full disk"]
    elif k == 1:
        i, a, b, port = f"SH-{r.randint(1000, 9999)}", r.randint(12, 40), r.randint(41, 90), r.choice(["Rotterdam", "Hamburg", "Valencia"])
        eta = date(2026, r.randint(1, 12), r.randint(1, 25)); old = eta - timedelta(r.randint(3, 6))
        text = (f"Shipment {i} left {port} with {a} pallets. The first plan listed {b} pallets and an arrival on {dfmt(old)}. Customs held two pallets for a paperwork check. "
                f"The carrier now expects arrival on {dfmt(eta)}, and the final load is {a} pallets. The buyer asked for a call on Monday. The insurance limit for this lane is ${est}00.")
        req, forb = [i, str(a), dfmt(eta), port], [str(b), dfmt(old)]
    else:
        i, who, found, fine = f"AUD-{r.randint(100, 999)}", r.choice(names), r.randint(3, 9), r.randrange(10, 60, 5)
        old = found + r.randint(4, 8); due = date(2026, r.randint(1, 11), r.randint(1, 25))
        text = (f"Audit {i} is signed off by {who}. Phase one listed {old} findings, but {old - found} were duplicates, so the final number is {found} findings. "
                f"The proposed penalty was ${fine + 25}k and the agreed penalty is ${fine}k. All fixes are due on {dfmt(due)}. The team will meet again in March. The auditor also praised the filing system.")
        req, forb = [i, str(found), f"${fine}k", dfmt(due)], [f"${fine + 25}k", str(old)]
    if any(has(" ".join(req), f) for f in forb): return make_summary(seed + 999)   # a stale figure must not hide inside a required fact
    body = f"Text:\n{text}"
    rv = "Facts: " + ", ".join(req) + "."
    return body, (req, forb, 40), rv
SUMMARY_FMT = "Write a summary of at most 40 words. It must contain the final facts of the text: ids, final numbers, dates and times. Do not repeat figures that the text later replaces."
SUMMARY = [make_summary(300 + i) for i in range(15)]
SUMMARY_EX = [make_summary(700 + i) for i in range(5)]

# ---- 5. question to SQL (SQLite; joins, HAVING, NOT EXISTS, correlated subqueries; results are compared, not the text)
SCHEMA_SQL = """CREATE TABLE departments(id INTEGER PRIMARY KEY, name TEXT, city TEXT);
CREATE TABLE employees(id INTEGER PRIMARY KEY, name TEXT, dept_id INTEGER, manager_id INTEGER, salary INTEGER, hired TEXT);  -- manager_id is NULL for top managers
CREATE TABLE projects(id INTEGER PRIMARY KEY, name TEXT, dept_id INTEGER, budget INTEGER, status TEXT);  -- status: active, paused or closed
CREATE TABLE assignments(emp_id INTEGER, project_id INTEGER, hours INTEGER);
CREATE TABLE tickets(id INTEGER PRIMARY KEY, project_id INTEGER, opened TEXT, closed TEXT, severity INTEGER);  -- closed is NULL while the ticket is open; dates are YYYY-MM-DD"""
def make_db():
    r = random.Random(11); db = sqlite3.connect(":memory:"); db.executescript(SCHEMA_SQL)
    db.executemany("INSERT INTO departments VALUES(?,?,?)", [(i + 1,) + d for i, d in enumerate([("Logistics", "Lyon"), ("Finance", "Porto"), ("Platform", "Lyon"), ("Research", "Graz"), ("Support", "Porto"), ("Legal", "Leeds")])])
    first = ["Ana", "Ben", "Chen", "Dara", "Eli", "Fay", "Gus", "Hana", "Ira", "Joy", "Kai", "Lou"]; names = [f"{a} {b}" for b in ["Ruiz", "Cole", "Wei", "Okoye", "Brandt", "Novak"] for a in first]
    r.shuffle(names)
    db.executemany("INSERT INTO employees VALUES(?,?,?,?,?,?)", [(i, n, (i - 1) % 6 + 1 if i <= 6 else r.randint(1, 6), None if i <= 6 else r.randint(1, 6), r.randrange(42000, 130000, 1000),
                    f"{r.randint(2015, 2025)}-{r.randint(1, 12):02d}-{r.randint(1, 28):02d}") for i, n in enumerate(names[:36], 1)])
    pn = ["Alder", "Birch", "Cedar", "Dune", "Ember", "Fjord", "Grove", "Harbor", "Iris", "Juniper", "Kestrel", "Lumen"]
    db.executemany("INSERT INTO projects VALUES(?,?,?,?,?)", [(i + 1, pn[i], r.randint(1, 6), r.randrange(20, 200, 5) * 1000, r.choice(["active", "active", "paused", "closed"])) for i in range(12)])
    asg = {(e, p) for e in range(1, 37) for p in r.sample(range(1, 13), r.choice([0, 1, 2, 2, 3, 4]))}
    db.executemany("INSERT INTO assignments VALUES(?,?,?)", [(e, p, r.randrange(10, 130, 5)) for e, p in sorted(asg)])
    tk = []
    for i in range(1, 61):
        o = f"202{r.randint(5, 6)}-{r.randint(1, 12):02d}-{r.randint(1, 28):02d}"; p = r.choice(range(1, 10))
        tk.append((i, p, o, None if r.random() < 0.4 else f"{o[:8]}{min(28, int(o[8:]) + r.randint(0, 12)):02d}", r.randint(1, 3)))
    db.executemany("INSERT INTO tickets VALUES(?,?,?,?,?)", tk)
    return db
DB_BYTES = make_db().serialize()
def run_sql(q):   # read-only, one statement, with an operation limit; returns sorted rows, or None when the SQL fails
    db = sqlite3.connect(":memory:"); db.deserialize(DB_BYTES); db.execute("PRAGMA query_only=1"); ops = [0]
    def stop(): ops[0] += 1; return ops[0] > 3000
    db.set_progress_handler(stop, 1000)
    try:
        if not re.match(r"\s*(select|with)\b", q, re.I): return None
        rows = [tuple(round(x, 2) if isinstance(x, float) else x for x in row) for row in db.execute(q).fetchall()]
        return sorted(rows, key=repr)
    except sqlite3.Error:
        return None
    finally:
        db.close()
SQL_Q = [
 ("Which departments have a total salary above 400000? One column: name.", "SELECT d.name FROM departments d JOIN employees e ON e.dept_id=d.id GROUP BY d.id HAVING SUM(e.salary)>400000"),
 ("Which employees have more than 250 assigned hours in total, over all their projects? One column: name.", "SELECT e.name FROM employees e JOIN assignments a ON a.emp_id=e.id GROUP BY e.id HAVING SUM(a.hours)>250"),
 ("Which projects have no tickets at all? One column: name.", "SELECT name FROM projects p WHERE NOT EXISTS (SELECT 1 FROM tickets t WHERE t.project_id=p.id)"),
 ("Which departments have no active project? One column: name.", "SELECT name FROM departments d WHERE NOT EXISTS (SELECT 1 FROM projects p WHERE p.dept_id=d.id AND p.status='active')"),
 ("Who has the most direct reports? One column: name of the manager.", "SELECT m.name FROM employees e JOIN employees m ON m.id=e.manager_id GROUP BY m.id ORDER BY COUNT(*) DESC LIMIT 1"),
 ("Which employees earn more than the average salary of their own department? One column: name.", "SELECT name FROM employees e WHERE salary>(SELECT AVG(salary) FROM employees x WHERE x.dept_id=e.dept_id)"),
 ("Which projects have at least 2 open tickets of severity 3? One column: name.", "SELECT p.name FROM projects p JOIN tickets t ON t.project_id=p.id WHERE t.closed IS NULL AND t.severity=3 GROUP BY p.id HAVING COUNT(*)>=2"),
 ("Which departments have more than 700 assigned hours in total, counting the hours of their employees on any project? One column: name.", "SELECT d.name FROM departments d JOIN employees e ON e.dept_id=d.id JOIN assignments a ON a.emp_id=e.id GROUP BY d.id HAVING SUM(a.hours)>700"),
 ("Which employees were hired before 2020-01-01 and work on no closed project? One column: name.", "SELECT name FROM employees e WHERE hired<'2020-01-01' AND NOT EXISTS (SELECT 1 FROM assignments a JOIN projects p ON p.id=a.project_id WHERE a.emp_id=e.id AND p.status='closed')"),
 ("In which cities do the departments have more than 8 employees in total? Two columns: city, number of employees.", "SELECT d.city, COUNT(*) FROM departments d JOIN employees e ON e.dept_id=d.id GROUP BY d.city HAVING COUNT(*)>8"),
 ("Which projects have a budget above the average budget of the projects of the same department? One column: name.", "SELECT name FROM projects p WHERE budget>(SELECT AVG(budget) FROM projects x WHERE x.dept_id=p.dept_id)"),
 ("Which employees have no assignment? One column: name.", "SELECT name FROM employees WHERE id NOT IN (SELECT emp_id FROM assignments)"),
 ("Which project has the most assigned hours in total? One column: name.", "SELECT p.name FROM projects p JOIN assignments a ON a.project_id=p.id GROUP BY p.id ORDER BY SUM(a.hours) DESC LIMIT 1"),
 ("For each project, how many tickets were closed within 7 days after they were opened (7 days included)? Only projects with at least 2 such tickets. Two columns: name, count.", "SELECT p.name, COUNT(*) FROM projects p JOIN tickets t ON t.project_id=p.id WHERE t.closed IS NOT NULL AND julianday(t.closed)-julianday(t.opened)<=7 GROUP BY p.id HAVING COUNT(*)>=2"),
 ("Which departments have at least one project and only active projects? One column: name.", "SELECT d.name FROM departments d WHERE EXISTS (SELECT 1 FROM projects p WHERE p.dept_id=d.id) AND NOT EXISTS (SELECT 1 FROM projects p WHERE p.dept_id=d.id AND p.status<>'active')"),
]
SQL_EX = [
 ("Which projects have a budget above 150000? One column: name.", "SELECT name FROM projects WHERE budget>150000"),
 ("How many employees work in each city? Two columns: city, count.", "SELECT d.city, COUNT(*) FROM departments d JOIN employees e ON e.dept_id=d.id GROUP BY d.city"),
 ("Which departments have no employee hired after 2022-12-31? One column: name.", "SELECT name FROM departments d WHERE NOT EXISTS (SELECT 1 FROM employees e WHERE e.dept_id=d.id AND e.hired>'2022-12-31')"),
 ("Which projects have more than 3 tickets? One column: name.", "SELECT p.name FROM projects p JOIN tickets t ON t.project_id=p.id GROUP BY p.id HAVING COUNT(*)>3"),
 ("Which employee earns the highest salary? One column: name.", "SELECT name FROM employees ORDER BY salary DESC LIMIT 1"),
]
SQL_FMT = "Reply with one SQLite SELECT statement only. No explanation."
def sql_body(q): return f"Schema:\n{SCHEMA_SQL}\n\nQuestion: {q}"

# ---- 6. log or code error diagnosis (hand-written; the loud error is a symptom, the cause is earlier or quieter)
DIAG_FMT = "Reply with one cause from the list only."
DIAG_HEAD = "Find the root cause: the earliest event that explains the later errors. Later errors are often only symptoms.\nCauses: " + ", ".join(CAUSES) + "\n\nLog:\n"
DIAG = [
 ("02:11:04 INFO backup job started\n02:11:09 ERROR write /var/data/wal.log: no space left on device\n02:11:10 ERROR db: could not flush WAL, read-only mode\n02:11:12 WARN api: 14 requests failed with 503\n02:11:15 ERROR healthcheck failed: db read-only", "disk_full"),
 ("10:02:01 WARN retry 1/3 to payments.partner.com\n10:02:02 ERROR tls: certificate has expired (notAfter=2026-10-31)\n10:02:02 ERROR upstream payments: 502\n10:02:09 WARN retry 3/3 failed, timeout after 5s\n10:02:10 ERROR checkout: payment step failed", "expired_certificate"),
 ("14:30:00 INFO worker started\n14:30:01 ERROR lookup db-primary.internal: no such host\n14:30:02 ERROR pool: 0 connections available\n14:30:05 ERROR request timeout after 5000 ms\n14:30:06 ERROR 12 jobs failed", "dns_failure"),
 ("09:15:10 INFO load test running, 300 users\n09:15:44 ERROR pool: timeout waiting for connection (size=20, in_use=20)\n09:15:45 ERROR request GET /orders: 504\n09:15:46 WARN retry storm: 300 retries per second\n09:15:50 INFO db cpu 8%, query p95 12 ms", "connection_pool_exhausted"),
 ("03:40:12 WARN heap usage 92%\n03:40:31 kernel: Out of memory: Killed process 8123 (worker)\n03:40:31 ERROR supervisor: worker exited with code 137\n03:40:33 ERROR queue: lag 12000 messages\n03:40:40 ERROR alert: no heartbeat from worker", "out_of_memory"),
 ("16:05:00 INFO deploy v8.2 finished, service user changed to svc-report\n16:05:07 ERROR open /srv/exports/report.csv: Permission denied\n16:05:08 ERROR job export-daily failed\n16:05:30 ERROR mail: attachment is empty, not sent\n16:06:00 WARN customer ticket #5521 opened", "permission_denied"),
 ("11:00:00 INFO starting billing-api\n11:00:01 ERROR KeyError: 'PAYMENT_API_URL'\n11:00:01 ERROR startup aborted\n11:00:30 ERROR healthcheck: connection refused\n11:00:31 ERROR orchestrator: restart loop, 5 restarts in 30 s", "missing_config"),
 ("Traceback (most recent call last):\n  File \"views.py\", line 41, in profile\n    return render(user.id, user.name)\nAttributeError: 'NoneType' object has no attribute 'id'\n# line 38: user = find_user(request.args.get('email'))\n# find_user returns None when the email is not found", "null_reference"),
 ("12:20:00 INFO migration 0042 skipped (already marked as applied)\n12:20:15 ERROR column orders.discount_pct does not exist\n12:20:15 ERROR GET /orders: 500\n12:20:40 ERROR 31 requests failed in the last minute\n12:21:00 ALERT error rate above 5%", "schema_mismatch"),
 ("00:05:00 INFO daily report for 2026-11-01 started\n00:05:03 INFO orders counted: 0 for the evening hours 19:00 to 24:00 (store time, UTC-5)\n00:05:03 WARN code: day = created_at.date()  # created_at is stored in UTC\n00:05:04 ERROR report total 18% below the ledger\n00:06:00 ERROR finance: ledger mismatch", "timezone_bug"),
 ("08:00:00.101 INFO worker-1 claimed job 77\n08:00:00.102 INFO worker-2 claimed job 77\n08:00:00.140 ERROR worker-2: duplicate key value violates unique constraint 'results_pkey'\n08:00:00.150 ERROR job 77 marked failed\n08:00:05 WARN customer charged twice", "race_condition"),
 ("13:10:00 INFO sync started, 5000 records\n13:10:20 ERROR HTTP 429 Too Many Requests from api.partner.com (Retry-After: 30)\n13:10:21 ERROR sync failed at record 1200\n13:10:22 WARN circuit breaker open\n13:10:25 ERROR dashboard: data is stale", "rate_limit"),
 ("20:01:10 INFO cron: nightly cleanup\n20:01:11 ERROR df: /var/log 100% used\n20:01:12 ERROR logrotate: cannot create /var/log/app.log.1: No space left on device\n20:01:14 ERROR app: log write failed, dropping requests\n20:01:20 ALERT 5xx rate 40%", "disk_full"),
 ("17:44:02 ERROR connect to cache-1:6379 timed out\n17:44:02 ERROR Temporary failure in name resolution: cache-1\n17:44:03 WARN fallback to database for sessions\n17:44:09 ERROR db cpu 99%\n17:44:20 ALERT p95 latency 9 s", "dns_failure"),
 ("Exception in thread \"main\" java.lang.NullPointerException: Cannot invoke \"Customer.getEmail()\" because \"customer\" is null\n    at Mailer.send(Mailer.java:57)\n    at Batch.run(Batch.java:22)\n// Mailer.java:55: Customer customer = repo.findById(id);  // returns null for unknown ids\n[ERROR] batch aborted at id=4412", "null_reference"),
]
DIAG_EX = [
 ("21:00:00 INFO job started\n21:00:02 ERROR HTTP 429 Too Many Requests, Retry-After: 10\n21:00:03 ERROR job failed\n21:00:04 ALERT queue growing", "rate_limit"),
 ("07:12:00 INFO deploy done\n07:12:03 ERROR open /etc/app/secret.key: Permission denied\n07:12:04 ERROR app cannot start\n07:12:10 ERROR healthcheck failed", "permission_denied"),
 ("19:30:00 WARN heap 95%\n19:30:20 kernel: Killed process 2210 (api) score 912\n19:30:21 ERROR 502 from gateway\n19:30:25 ALERT api down", "out_of_memory"),
 ("05:00:00 INFO start\n05:00:01 ERROR env var DATABASE_URL is not set\n05:00:01 ERROR exit code 1\n05:00:30 ERROR restart loop", "missing_config"),
 ("15:00:00 INFO TLS handshake to api.shop.com\n15:00:01 ERROR x509: certificate has expired\n15:00:02 ERROR order sync failed\n15:00:30 ALERT sync lag 20 min", "expired_certificate"),
]

def mk(uc, i, body, fmt, rv, gold): return dict(id=f"{uc}-{i}", uc=uc, idx=i, body=body, fmt=fmt, rv=rv, gold=gold)
def build_items():
    its = []
    for i, (t, l) in enumerate(TRIAGE): its.append(mk("triage", i, TRIAGE_RULES + "\n\nEmail:\n" + t, TRIAGE_FMT, l, l))
    for i, (b, a) in enumerate(ACTIONS): its.append(mk("actions", i, b, ACT_FMT, a, frozenset((x["owner"], x["due"]) for x in a)))
    for i, (b, (a, k)) in enumerate(DOCQA): its.append(mk("docqa", i, b, "", str(int(a)) if k == "num" and a == int(a) else str(a), (a, k)))
    for i, (b, g, rv) in enumerate(SUMMARY): its.append(mk("summary", i, b, SUMMARY_FMT, rv, g))
    for i, (q, s) in enumerate(SQL_Q): its.append(mk("sql", i, sql_body(q), SQL_FMT, s, run_sql(s)))
    for i, (t, l) in enumerate(DIAG): its.append(mk("diagnosis", i, DIAG_HEAD + t, DIAG_FMT, l, l))
    return its
def build_examples():
    ex = {uc: [] for uc in UCS}
    for t, l in TRIAGE_EX: ex["triage"].append(dict(body=TRIAGE_RULES + "\n\nEmail:\n" + t, rv=l))
    for b, a in ACTIONS_EX: ex["actions"].append(dict(body=b, rv=a))
    for b, (a, k) in DOCQA_EX: ex["docqa"].append(dict(body=b, rv=str(int(a)) if k == "num" and a == int(a) else str(a)))
    for b, g, rv in SUMMARY_EX: ex["summary"].append(dict(body=b, rv=rv))
    for q, s in SQL_EX: ex["sql"].append(dict(body=sql_body(q), rv=s))
    for t, l in DIAG_EX: ex["diagnosis"].append(dict(body=DIAG_HEAD + t, rv=l))
    return ex
ITEMS, EX = build_items(), build_examples()
for it in ITEMS: it["split"] = "dev" if it["idx"] % 2 == 0 else "test"   # choose settings on dev, measure on test
assert len(ITEMS) == 90 and len({it["id"] for it in ITEMS}) == 90 and all(len(v) == 5 for v in EX.values())
assert not {it["body"] for it in ITEMS} & {e["body"] for v in EX.values() for e in v}, "an example equals a test item"
assert all(it["gold"] for it in ITEMS if it["uc"] == "sql") and all(0 < len(it["gold"]) < 20 for it in ITEMS if it["uc"] == "sql")
print(pd.Series([it["uc"] for it in ITEMS]).value_counts().to_dict(), "| dev items:", sum(it["split"] == "dev" for it in ITEMS))
for u in UCS: it = next(x for x in ITEMS if x["uc"] == u); print(f"{u:9s}", it["body"].splitlines()[-1][:95], "->", str(it["rv"])[:45])

In [ ]:
# 3. Graders and prompt builder. Every grader is plain Python. Each is tested on right and wrong replies, for every item.
def clean(text):   # drop thinking text, and keep what follows the last "ANSWER:" (chain-of-thought replies end with it)
    t = text.split("</think>")[-1]
    return (t.rsplit("ANSWER:", 1)[1] if "ANSWER:" in t else t).strip()

def parse(it, text, structured):   # reply text -> value, or None when the reply cannot be read (a format error)
    t, uc = clean(text), it["uc"]
    if not t: return None
    if structured:
        try: v = json.loads(t[t.index("{"): t.rindex("}") + 1])[FIELD[uc]]
        except (ValueError, KeyError, TypeError): return None
    else:
        v = t
    if uc in ("triage", "diagnosis"):
        s = re.sub(r"^(label|cause|answer)\s*:\s*", "", str(v).strip().strip("`'\"*. \n").lower()).strip("`'\"*. \n")
        return s if s in (TRIAGE_LABELS if uc == "triage" else CAUSES) else None
    if uc == "actions":
        try:
            if isinstance(v, str): v = json.loads(re.search(r"[\[{].*[\]}]", v, re.S).group(0))
            v = v["actions"] if isinstance(v, dict) else v
            return frozenset((x["owner"].strip(), x["due"].strip()) for x in v)
        except (AttributeError, ValueError, KeyError, TypeError): return None
    s = str(v).strip()
    if uc == "docqa":
        if it["gold"][1] == "date": m = re.search(r"\d{4}-\d{2}-\d{2}", s); return m.group(0) if m else None
        m = re.search(r"-?\d[\d,]*\.?\d*", s); return float(m.group(0).replace(",", "").rstrip(".")) if m else None
    if uc == "sql":
        m = re.search(r"```(?:sql)?\s*(.*?)```", s, re.S); return run_sql((m.group(1) if m else s).strip().rstrip(";"))
    return s   # summary

def check(it, v):
    uc, g = it["uc"], it["gold"]
    if uc == "docqa": return abs(v - g[0]) < 0.005 if g[1] == "num" else v == g[0]
    if uc == "summary": return len(v.split()) <= g[2] and all(has(v, f) for f in g[0]) and not any(has(v, f) for f in g[1])
    return v == g

def messages_for(it, tech, system=None):   # one prompt per technique; few-shot examples are earlier chat turns
    user, extra, ms = it["body"] + ("\n\n" + it["fmt"] if it["fmt"] else ""), {}, []
    if tech == "system_rules": system = SYSTEM[it["uc"]]
    if system: ms.append({"role": "system", "content": system})
    if tech.startswith("few_"):
        for e in EX[it["uc"]][:int(tech[4:])]:
            ms += [{"role": "user", "content": e["body"] + ("\n\n" + it["fmt"] if it["fmt"] else "")}, {"role": "assistant", "content": plain(it["uc"], e["rv"])}]
    if tech == "cot": user += "\n\nThink step by step in a few short lines. Then write the final answer on the last line as 'ANSWER: ' followed by the answer in the format above."
    if tech == "structured":
        user = it["body"] + "\n\nAnswer in the JSON format of the schema."
        extra = {"response_format": {"type": "json_schema", "json_schema": {"name": "answer", "strict": True, "schema": SCHEMAS[it["uc"]]}}}
    return ms + [{"role": "user", "content": user}], extra

def plain(uc, rv): return json.dumps({"actions": rv}) if uc == "actions" else rv
STR = {"type": "string"}
SCHEMAS = {uc: {"type": "object", "properties": {FIELD[uc]: STR}, "required": [FIELD[uc]], "additionalProperties": False} for uc in UCS}
SCHEMAS["triage"]["properties"]["label"] = {"type": "string", "enum": TRIAGE_LABELS}
SCHEMAS["diagnosis"]["properties"]["cause"] = {"type": "string", "enum": CAUSES}
SCHEMAS["actions"]["properties"]["actions"] = {"type": "array", "items": {"type": "object", "properties": {"owner": STR, "due": STR}, "required": ["owner", "due"], "additionalProperties": False}}
SYSTEM = {  # fixed before any result was seen
 "triage": "You triage support email. Apply the label definitions exactly. If two labels fit, apply the stated order. Reply with the label only.",
 "actions": "You extract action items. Include only tasks that a named person agreed to do. Compute each date from the meeting date. Reply with JSON only.",
 "docqa": "You answer questions about a document. Use the newest valid fact for the date asked. Check every condition. Reply with the answer only.",
 "summary": "You write short summaries. Keep every final id, number, date and time. Never repeat a figure that the text replaces later. Count your words.",
 "sql": "You write SQLite queries. Check each join, each HAVING and each NOT EXISTS against the question. Reply with one SELECT statement only.",
 "diagnosis": "You diagnose software failures. Name the earliest cause, not the loudest error. Reply with one cause from the list only.",
}
TECHS = ["zero_shot", "few_1", "few_3", "few_5", "cot", "system_rules", "structured"]

def grade_reply(it, text, structured=False):
    v = parse(it, text, structured)
    return None if v is None else check(it, v)

def structured_reply(it): return json.dumps({FIELD[it["uc"]]: it["rv"]})
def wrong_value(it):   # a plausible wrong answer for each use case
    uc, rv = it["uc"], it["rv"]
    if uc == "triage": return TRIAGE_LABELS[(TRIAGE_LABELS.index(rv) + 1) % 6]
    if uc == "diagnosis": return CAUSES[(CAUSES.index(rv) + 1) % 12]
    if uc == "actions": return rv[:-1] + [dict(rv[-1], due=(date.fromisoformat(rv[-1]["due"]) + timedelta(1)).isoformat())]
    if uc == "docqa": return str(float(rv) + 1) if re.fullmatch(r"-?[\d.]+", rv) else "2000-01-01"
    if uc == "summary": return "Facts: " + ", ".join(it["gold"][0][:-1] + it["gold"][1][:1]) + "."
    return "SELECT name FROM employees"
def selftest_graders():
    for it in ITEMS:
        right, wrong = plain(it["uc"], it["rv"]), plain(it["uc"], wrong_value(it))
        assert grade_reply(it, right) is True, ("right reply rejected", it["id"])
        assert grade_reply(it, "```\n" + right + "\n```" if it["uc"] != "sql" else "```sql\n" + right + ";\n```") is True, ("fenced reply", it["id"])
        assert grade_reply(it, "Step 1 ... \nANSWER: " + right) is True and grade_reply(it, "<think>x</think>" + right) is True, ("answer line", it["id"])
        assert grade_reply(it, json.dumps({FIELD[it["uc"]]: it["rv"]}), True) is True, ("structured right", it["id"])
        assert grade_reply(it, wrong) is False, ("wrong reply accepted", it["id"], wrong)
        assert grade_reply(it, json.dumps({FIELD[it["uc"]]: wrong_value(it)}), True) is False, ("structured wrong accepted", it["id"])
        assert grade_reply(it, "I am not sure.") in (None, False) and grade_reply(it, "") is None, ("garbage", it["id"])
    for it in (x for x in ITEMS if x["uc"] == "sql"):   # SQL: a write, a syntax error and a near miss must all fail
        assert grade_reply(it, "DELETE FROM employees") is None and grade_reply(it, "SELEC name FROM") is None
    s = next(x for x in ITEMS if x["uc"] == "summary"); good = " ".join(s["gold"][0])
    assert grade_reply(s, good) is True and grade_reply(s, good + " " + s["gold"][1][0]) is False and grade_reply(s, good + " word" * 40) is False
    a = next(x for x in ITEMS if x["uc"] == "actions"); assert grade_reply(a, plain("actions", a["rv"][:-1])) is False   # a missing action is wrong
    # the few-shot examples must also pass their own grader
    for u in UCS:
        for e in EX[u]:
            fake = next(i for i in ITEMS if i["uc"] == u); ex_it = dict(fake, rv=e["rv"], body=e["body"])
            if u == "docqa": ex_it["gold"] = (float(e["rv"]) if not re.match(r"\d{4}-", e["rv"]) else e["rv"], "num" if not re.match(r"\d{4}-", e["rv"]) else "date")
            if u == "summary": ex_it["gold"] = next(g for b, g, rv in SUMMARY_EX if rv == e["rv"])
            if u == "sql": ex_it["gold"] = run_sql(e["rv"]); assert ex_it["gold"] is not None
            if u == "actions": ex_it["gold"] = frozenset((x["owner"], x["due"]) for x in e["rv"])
            if u in ("triage", "diagnosis"): ex_it["gold"] = e["rv"]
            assert grade_reply(ex_it, plain(u, e["rv"])) is True, ("example fails its grader", u)
    print("graders ok: 90 items x (right, fenced, answer line, structured right, wrong, structured wrong, garbage); 30 few-shot examples pass")
selftest_graders()
# baselines: the best constant reply per use case, and random choice among the labels
CONST = dict(triage=TRIAGE_LABELS, diagnosis=CAUSES, actions=['{"actions": []}'], docqa=["0"], summary=["Summary."], sql=["SELECT 1"])
BASELINE = {}
for uc in UCS:
    its = [i for i in ITEMS if i["uc"] == uc]
    BASELINE[uc] = (max(float(np.mean([grade_reply(i, c) is True for i in its])) for c in CONST[uc]), 1 / len(CONST[uc]) if uc in ("triage", "diagnosis") else 0.0)
print("baselines (best constant reply, random label):", {k: tuple(round(x, 2) for x in v) for k, v in BASELINE.items()})

In [ ]:
# 4. Part A: runner (every outcome is kept) and the zero-shot results of 6 use cases x 15 items on two models
ROWS, OUTCOMES = [], ["wrong", "format_error", "empty", "cut", "error", "not_reached"]
def judge(it, tech, r):
    if not r["ok"]: return "not_reached" if r["error"] in ("cap", "aborted", "no key") else "error"
    if r["finish"] == "length": return "cut"                  # the reply was cut by max_tokens: an outcome of its own
    if not clean(r["content"]): return "empty"
    g = grade_reply(it, r["content"], tech == "structured")
    return "format_error" if g is None else ("correct" if g else "wrong")

def run(model, tech, items, rep=1, system=None, tag=None):
    def one(it):
        ms, extra = messages_for(it, tech, system)
        return chat(model, ms, extra, max_tokens=700 if tech == "cot" else 500, deadline=DEADLINE)
    rows = []
    for it, r in zip(items, pmap(one, items, DEADLINE)):
        r = r or dict(ok=False, error="not_reached", content="", finish="", ptok=0, ctok=0, latency_s=float("nan"))
        rows.append(dict(model=model.split("/")[-1], tech=tag or tech, rep=rep, uc=it["uc"], id=it["id"], split=it["split"], outcome=judge(it, tech, r),
                         latency_s=r["latency_s"], ptok=r["ptok"], ctok=r["ctok"], text=r["content"][:300], err=r["error"]))
    ROWS.extend(rows)
    return pd.DataFrame(rows)

def table(df, by):   # success is over all scheduled items: errors and not-reached items count as not correct and are listed
    out = []
    for k, g in df.groupby(by, sort=False):
        c = int((g.outcome == "correct").sum()); lo, hi = wilson(c, len(g))
        out.append({**dict(zip(by, k if isinstance(k, tuple) else (k,))), "n": len(g), "success": c / len(g), "lo": lo, "hi": hi,
                    **{o: int((g.outcome == o).sum()) for o in OUTCOMES}, "ptok": g.ptok.mean(), "ctok": g.ctok.mean(),
                    "p50_s": g.latency_s.median(), "p95_s": g.latency_s.quantile(0.95)})
    return pd.DataFrame(out)

def short(m): return m.split("/")[-1]
def api_answered(): return any(r["outcome"] not in ("error", "not_reached") for r in ROWS)
def part_a():
    global CALL_LIMIT; CALL_LIMIT = 1100   # keeps 400 calls for part C
    if not KEY: print("no NVIDIA_API_KEY: parts A to C are skipped, parts D to F use FALLBACK rates"); return
    for m in (SMALL, MID): run(m, "zero_shot", ITEMS)
    if not api_answered(): print("the API did not answer: parts B and C are skipped, parts D to F use FALLBACK rates"); return
    df = pd.DataFrame(ROWS); t = table(df, ["model", "uc"])
    t["const_base"] = t.uc.map(lambda u: BASELINE[u][0]); t["random_base"] = t.uc.map(lambda u: BASELINE[u][1])
    print(t.round(2).to_string(index=False)); t.round(3).to_csv(f"{OUT_DIR}/prompting_a_use_cases.csv", index=False)
    print(table(df, ["model"]).round(2).to_string(index=False))
part("A use cases, zero-shot, 2 models", part_a, 400)

## Part B. Prompt techniques

Question: which technique raises success on the same 90 items, and what does it cost in format errors, tokens and time?

Seven techniques run on the small model: zero-shot, few-shot with 1, 3 and 5 examples, chain of thought, a system prompt with rules, and structured output through `response_format`. Zero-shot and chain of thought run twice, because the API gives different answers between runs even at temperature 0. Chain of thought also runs on the mid model. The pick is chosen on the 48 dev items. It is measured on the 42 test items. Each technique is compared with zero-shot on the same items with an exact McNemar test.

In [ ]:
# 5. Part B: prompt techniques on the small model (all 90 items), plus repeats and a second model. Pick on dev items, measure on test items.
def vec(model, tech, rep=1):   # 0/1 success per item, in item order; items with no row count as 0
    df = pd.DataFrame(ROWS); g = df[(df.model == short(model)) & (df.tech == tech) & (df.rep == rep)].set_index("id").outcome
    return np.array([int(g.get(it["id"]) == "correct") for it in ITEMS])
DEV = np.array([it["split"] == "dev" for it in ITEMS])
PICK = {}
def part_b():
    if not api_answered(): return print("skipped: no answer in part A")
    for t in TECHS[1:]: run(SMALL, t, ITEMS)
    for t in ("zero_shot", "cot"): run(SMALL, t, ITEMS, rep=2)
    run(MID, "cot", ITEMS)
    df = pd.DataFrame(ROWS); tab = table(df, ["model", "tech", "rep"])
    tab["const_base"] = np.mean([BASELINE[u][0] for u in UCS])   # best constant reply, averaged over the 6 equal use cases
    print(tab.round(2).to_string(index=False)); tab.round(3).to_csv(f"{OUT_DIR}/prompting_b_techniques.csv", index=False)
    pd.DataFrame(ROWS).to_csv(f"{OUT_DIR}/prompting_raw_rows.csv", index=False)
    z = vec(SMALL, "zero_shot"); rows = []
    for t in TECHS:
        v = vec(SMALL, t); x, y, p = mcnemar(z, v)
        rows.append(dict(tech=t, dev=v[DEV].mean(), test=v[~DEV].mean(), all=v.mean(), delta_all=v.mean() - z.mean(), only_zero=x, only_tech=y, mcnemar_p=p))
    cmp = pd.DataFrame(rows); print(cmp.round(3).to_string(index=False)); cmp.round(3).to_csv(f"{OUT_DIR}/prompting_b_vs_zero_shot.csv", index=False)
    best = cmp[cmp.tech != "zero_shot"].sort_values("dev", ascending=False).iloc[0]; zt = cmp[cmp.tech == "zero_shot"].iloc[0]
    PICK.update(tech=best.tech, dev=best.dev, test=best.test, zero_test=zt.test)
    x, y, p = mcnemar(z[~DEV], vec(SMALL, best.tech)[~DEV])
    print(f"picked on dev: {best.tech} (dev {best.dev:.2f}); on the {int((~DEV).sum())} test items: {best.test:.2f} against zero-shot {zt.test:.2f}, "
          f"only zero-shot right {x}, only {best.tech} right {y}, exact McNemar p={p:.3f}")
    for t in ("zero_shot", "cot"):   # run-to-run noise at temperature 0: same prompts, second run
        a, b = vec(SMALL, t, 1), vec(SMALL, t, 2)
        print(f"repeat {t:10s}: run 1 {a.mean():.2f}, run 2 {b.mean():.2f}, items whose result changed {int((a != b).sum())} of {len(a)}")
    s1 = tab[(tab.model == short(SMALL)) & (tab.rep == 1)]
    plt.figure(figsize=(7, 3.5)); plt.bar(s1.tech, s1.success, yerr=[s1.success - s1.lo, s1.hi - s1.success], capsize=3)
    plt.ylabel("success (Wilson 95%)"); plt.xticks(rotation=30); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/prompting_b_techniques.png", dpi=110); plt.close()
part("B prompt techniques", part_b, 1100)

## Part C. Prompt optimization

Question: when a large model rewrites the system prompt from the errors on dev items, does the gain hold on held-out items?

Round 0 is no system prompt. In each of 4 rounds the large model sees up to 12 dev errors (2 per use case at most) with the right answers. It writes a new system prompt. The small model then runs on the dev items and on the test items. The round with the best dev score is the pick. Its test score is the honest number. A dev gain that is larger than the test gain is overfitting to the dev errors.

In [ ]:
# 6. Part C: automatic prompt optimization. Dev errors go to the large model, which writes the next system prompt. Four rounds.
OPT_ASK = ("You improve the system prompt of a small language model. The model does six kinds of work: email triage, action items from meeting notes, "
           "document questions, summaries that keep given facts, question to SQL, and log diagnosis. Below are the current system prompt and tasks that the model got wrong, "
           "with its reply and the right answer. Write a better system prompt of at most 200 words. Give general rules that fix these kinds of mistakes. "
           "Do not copy facts of single tasks. Reply with the new system prompt only.")
def part_c():
    if not PICK: return print("skipped: part B did not finish")
    global CALL_LIMIT; CALL_LIMIT = MAX_CALLS
    df = pd.DataFrame(ROWS); last = df[(df.model == short(SMALL)) & (df.tech == "zero_shot") & (df.rep == 1)]
    by_id = {it["id"]: it for it in ITEMS}; dev, test = [i for i in ITEMS if i["split"] == "dev"], [i for i in ITEMS if i["split"] == "test"]
    prompts, hist = [None], [dict(round=0, dev=(last[last.split == "dev"].outcome == "correct").mean(), test=(last[last.split == "test"].outcome == "correct").mean(), note="no system prompt")]
    for rd in range(1, 5):
        errs = last[(last.split == "dev") & (last.outcome != "correct")]
        errs = pd.concat([g.head(2) for _, g in errs.groupby("uc")]).head(12)   # up to 2 errors per use case, 12 in all
        shown = "\n\n".join(f"[{e.uc}] task: {by_id[e.id]['body'][-450:]}\nmodel reply: {e.text[:150]}\nright answer: {plain(e.uc, by_id[e.id]['rv'])[:150]}" for e in errs.itertuples())
        r = chat(BIG, [{"role": "user", "content": f"{OPT_ASK}\n\nCurrent system prompt:\n{prompts[-1] or '(none)'}\n\nTasks the model got wrong:\n{shown}"}], max_tokens=500, deadline=DEADLINE)
        new = r["content"].strip().strip("`")[:1500] if r["ok"] and r["finish"] != "length" else ""
        note = "new prompt" if new else "optimizer failed, kept the old prompt (" + (r["error"] or r["finish"] or "empty") + ")"
        prompts.append(new or prompts[-1])
        d = run(SMALL, "zero_shot", dev, system=prompts[-1], tag=f"opt_r{rd}"); t = run(SMALL, "zero_shot", test, system=prompts[-1], tag=f"opt_r{rd}")
        last = d; hist.append(dict(round=rd, dev=(d.outcome == "correct").mean(), test=(t.outcome == "correct").mean(), note=note))
    h = pd.DataFrame(hist); h["dev_n"], h["test_n"] = len(dev), len(test); h["test_lo"], h["test_hi"] = zip(*[wilson(round(x * len(test)), len(test)) for x in h.test])
    h["gap_dev_minus_test"] = h.dev - h.test; h["const_base"] = np.mean([BASELINE[u][0] for u in UCS]); print(h.round(3).to_string(index=False)); h.round(3).to_csv(f"{OUT_DIR}/prompting_c_optimization.csv", index=False)
    json.dump(prompts, open(f"{OUT_DIR}/prompting_c_prompts.json", "w"), indent=1)
    b = int(h.dev.values.argmax())   # the round is chosen on dev, then measured on test
    tag = "zero_shot" if b == 0 else f"opt_r{b}"; x, y, p = mcnemar(vec(SMALL, "zero_shot")[~DEV], vec(SMALL, tag)[~DEV])
    print(f"round {b} has the best dev score ({h.dev[b]:.2f}). Its test score is {h.test[b]:.2f} against {h.test[0]:.2f} with no prompt; "
          f"only no-prompt right {x}, only optimized right {y}, exact McNemar p={p:.3f}. Dev gain {h.dev[b] - h.dev[0]:+.2f}, test gain {h.test[b] - h.test[0]:+.2f}.")
    OPT.update(round=b, dev_gain=h.dev[b] - h.dev[0], test_gain=h.test[b] - h.test[0])
OPT = {}
part("C prompt optimization", part_c, 900)

## Part D. Offline experiments

Question: how many items does a golden set need, and when does a change that wins overall still lose on a slice?

No API call here. It uses the paired results of part B: zero-shot against the pick. A paired bootstrap gives an interval for the gain. The power table comes from a simulation of the exact McNemar test with the measured share of items where the two prompts differ. A simulation with no gain must give at most 5% false alarms (an assert). The offline gate says go or stop, from the interval. A synthetic slice example has a known truth that the code asserts.

In [ ]:
# 7. Part D: offline experiments (no API needed). Inputs are measured when part B ran, and stated FALLBACK numbers when it did not.
FALLBACK = dict(zero=dict(triage=0.55, actions=0.35, docqa=0.45, summary=0.40, sql=0.40, diagnosis=0.50), gain=0.06, c1=0.08,   # stated assumptions
                ctok_a=40, ctok_b=260, lat_a=0.5, lat_b=2.5)
def load_inputs():
    df = pd.DataFrame(ROWS); uc = np.array([it["uc"] for it in ITEMS])
    z = df[(df.model == short(SMALL)) & (df.tech == "zero_shot") & (df.rep == 1)] if len(df) else df
    if len(z) and PICK and (~z.outcome.isin(["error", "not_reached"])).mean() >= 0.8:   # the API ran
        t = df[(df.model == short(SMALL)) & (df.tech == PICK["tech"]) & (df.rep == 1)]
        return dict(source=f"MEASURED (small model, zero_shot against {PICK['tech']})", a=vec(SMALL, "zero_shot"), b=vec(SMALL, PICK["tech"]), uc=uc,
                    lat_a=z.latency_s.dropna().values, lat_b=t.latency_s.dropna().values, ctok_a=z.ctok.mean(), ctok_b=t.ctok.mean())
    r = np.random.default_rng(SEED); p = np.array([FALLBACK["zero"][u] for u in uc]); a = (r.random(90) < p).astype(int)
    c0 = (FALLBACK["gain"] + FALLBACK["c1"] * p.mean()) / (1 - p.mean())   # P(B right | A wrong), so that the mean gain equals FALLBACK gain
    b = np.where(a == 1, r.random(90) >= FALLBACK["c1"], r.random(90) < c0).astype(int)
    assert abs(a.mean() - p.mean()) < 0.12 and abs((b - a).mean() - FALLBACK["gain"]) < 0.12, "fallback draw is far from its stated rates"
    return dict(source="FALLBACK RATES (the API did not run; numbers below are stated assumptions, not measurements)", a=a, b=b, uc=uc,
                lat_a=r.lognormal(np.log(FALLBACK["lat_a"]), 0.4, 90), lat_b=r.lognormal(np.log(FALLBACK["lat_b"]), 0.4, 90), ctok_a=FALLBACK["ctok_a"], ctok_b=FALLBACK["ctok_b"])

def crit_table(M):   # crit[m] = largest k so that exact two-sided McNemar rejects when min(only-A, only-B) <= k and there are m discordant items
    c = np.full(M + 1, -1)
    for m in range(1, M + 1):
        ks = np.arange(m // 2 + 1); ok = 2 * stats.binom.cdf(ks, m, 0.5) <= 0.05
        if ok.any(): c[m] = ks[ok].max()
    return c
CRIT = crit_table(1200)
def power(n, delta, d, sims=3000, seed=SEED):   # chance that exact McNemar finds a true gain delta; d = share of items where the two prompts differ
    d = max(d, abs(delta)); pb, pa = (d + delta) / 2, (d - delta) / 2   # pb: only the new prompt is right, pa: only the old one
    k = np.random.default_rng(seed).multinomial(n, [pb, pa, 1 - pb - pa], size=sims); m = k[:, 0] + k[:, 1]
    return float(((np.minimum(k[:, 0], k[:, 1]) <= CRIT[m]) & (m > 0)).mean())

def part_d():
    D = load_inputs(); a, b, uc = D["a"], D["b"], D["uc"]; print("inputs:", D["source"])
    rng = np.random.default_rng(SEED); idx = rng.integers(0, 90, (5000, 90)); idx2 = rng.integers(0, 90, (5000, 90))
    paired = (b - a)[idx].mean(1); unpaired = b[idx].mean(1) - a[idx2].mean(1); x, y, p = mcnemar(a, b)
    d = float((a != b).mean()); lo, hi = np.percentile(paired, [2.5, 97.5]); ulo, uhi = np.percentile(unpaired, [2.5, 97.5])
    pg = (pd.DataFrame([dict(items=90, old=a.mean(), new=b.mean(), gain=b.mean() - a.mean(), paired_lo=lo, paired_hi=hi, unpaired_lo=ulo, unpaired_hi=uhi,
                             only_old=x, only_new=y, discord=d, mcnemar_p=p)]).round(3)); print(pg.to_string(index=False)); pg.to_csv(f"{OUT_DIR}/prompting_d_paired.csv", index=False)
    print(f"interval width: paired {hi - lo:.3f}, unpaired {uhi - ulo:.3f}; the paired one is " + ("narrower, because both prompts see the same items" if hi - lo < uhi - ulo else "not narrower in this run"))
    gate = lo > 0
    print("offline gate (lower end of the paired interval above 0):", "PASS, go to an online test" if gate else f"FAIL, lower end is {lo:+.3f}: the offline data does not prove a gain")
    ns, deltas = [30, 60, 90, 150, 250, 400, 600, 1000], [0.02, 0.05, 0.10, 0.15]
    pw = pd.DataFrame({f"gain {g:.2f}": [power(n, g, d) for n in ns] for g in deltas}, index=pd.Index(ns, name="items")); print(f"power of exact McNemar (discordance {d:.2f}, {3000} sims)"); print(pw.round(2).to_string())
    pw.round(3).to_csv(f"{OUT_DIR}/prompting_d_power.csv")
    need = {g: next((n for n in ns if pw.loc[n, f"gain {g:.2f}"] >= 0.8), None) for g in deltas}
    print("smallest golden set with power >= 0.8:", {f"gain {g:.2f}": (n if n else ">1000") for g, n in need.items()})
    assert power(250, 0.0, d) <= 0.06 and power(1000, 0.10, d) > 0.95, "power simulation does not match the exact test"   # truth: no gain -> at most 5% false alarms
    sl = pd.DataFrame([dict(use_case=u, n=int((uc == u).sum()), old=a[uc == u].mean(), new=b[uc == u].mean(), delta=(b - a)[uc == u].mean()) for u in UCS])
    print(sl.round(2).to_string(index=False)); sl.round(3).to_csv(f"{OUT_DIR}/prompting_d_slices.csv", index=False); lost = sl[sl.delta < 0]
    print(f"overall change {b.mean() - a.mean():+.3f}; slices that lose: {list(lost.use_case) or 'none in this run'}")
    # a slice reversal with a known truth: the new prompt wins on 80 items and loses on 20, and wins overall
    big, small = (80, 0.70, 0.78), (20, 0.80, 0.60); ov = [(big[0] * big[i] + small[0] * small[i]) / 100 for i in (1, 2)]
    assert ov[1] > ov[0] and small[2] < small[1], "slice example is not a reversal"
    print(f"SYNTHETIC slice example: overall {ov[0]:.3f} -> {ov[1]:.3f} (wins), slice of 20 items {small[1]:.2f} -> {small[2]:.2f} (loses). "
          f"To see a loss of {small[1] - small[2]:.2f} in that slice the golden set needs {next((n for n in ns if power(n, 0.2, 0.3) >= 0.8), '>1000')} items in it")
part("D offline experiments", part_d, 300)

## Part E. Online A/B test, simulated

Question: what goes wrong in a real A/B test, and how do shadow and canary compare with it?

No API call here. Users are drawn from the success rates of part D. Every simulation has a truth that the code asserts: the power at the planned sample size, the false positive rate against an independent Gaussian simulation, the sample ratio mismatch alarm, the novelty decay and the Simpson reversal. E5 uses the measured latency and token numbers as guardrail metrics. E7 uses a planted bad version, so you can see how many users each method exposes.

In [ ]:
# 8. Part E: online A/B test, simulated (no API). Users are drawn from the success rates of part D. Fixed seeds. Every simulator truth is asserted.
def zp(ca, na, cb, nb):   # two-sided p of the two-proportion z-test; arrays allowed
    pp = (ca + cb) / (na + nb); se = np.sqrt(np.maximum(pp * (1 - pp) * (1 / na + 1 / nb), 1e-12))
    return 2 * stats.norm.sf(np.abs((cb / nb - ca / na) / se))

def part_e():
    D = load_inputs(); a, b = D["a"], D["b"]; pa, pb = float(np.clip(a.mean(), 0.15, 0.9)), float(np.clip(b.mean(), 0.01, 0.99))   # clipped so that the simulations stay meaningful
    print("inputs:", D["source"], f"| old success {pa:.3f}, new success {pb:.3f}"); rng = np.random.default_rng(SEED); S = 6000
    # E1. sample size for the smallest gain worth shipping (MDE), checked by simulation
    mde = 0.05; q = pa + mde; n = math.ceil((1.959964 + 0.841621) ** 2 * (pa * (1 - pa) + q * (1 - q)) / mde ** 2)
    pw = float((zp(rng.binomial(n, pa, S), n, rng.binomial(n, q, S), n) < 0.05).mean()); assert 0.76 < pw < 0.84, ("power is not 0.8", pw)
    print(f"E1 sample size: to see a gain of {mde:.2f} from {pa:.2f} with power 0.80 at alpha 0.05 you need {n} users per arm; simulated power at that size {pw:.3f}")
    # E2. false positives of A/A tests (no true difference): one look, 10 looks, and 10 looks with a Bonferroni-corrected threshold
    def fp(peeks, alpha=0.05):
        seg = n // peeks; m = seg * np.arange(1, peeks + 1)
        return float((zp(rng.binomial(seg, pa, (S, peeks)).cumsum(1), m, rng.binomial(seg, pa, (S, peeks)).cumsum(1), m) < alpha).any(1).mean())
    def theory(peeks):   # idealised Gaussian walk: z at look k is S_k / sqrt(k); an independent simulation of the same rule
        z = np.random.default_rng(1).standard_normal((200000, peeks)).cumsum(1) / np.sqrt(np.arange(1, peeks + 1)); return float((np.abs(z) > 1.959964).any(1).mean())
    t = pd.DataFrame([dict(looks=k, false_positive=fp(k), gaussian_theory=theory(k), bonferroni=fp(k, 0.05 / k)) for k in (1, 2, 5, 10, 20)]); print("E2 false positive rate of A/A tests (should be 0.05):"); print(t.round(3).to_string(index=False))
    t.round(4).to_csv(f"{OUT_DIR}/prompting_e_peeking.csv", index=False); f1, f10 = t.false_positive[0], t.false_positive[3]
    assert abs(f1 - 0.05) < 0.015 and f10 > 0.12 and abs(f10 - t.gaussian_theory[3]) < 0.03 and t.bonferroni[3] < 0.07, ("peeking simulation disagrees with theory", t)
    plt.figure(figsize=(5, 3)); plt.plot(t.looks, t.false_positive, "o-", label="uncorrected"); plt.plot(t.looks, t.bonferroni, "s-", label="Bonferroni"); plt.axhline(0.05, color="k", ls=":")
    plt.xlabel("number of looks"); plt.ylabel("false positive rate"); plt.legend(); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/prompting_e_peeking.png", dpi=110); plt.close()
    # E3. sample ratio mismatch: a planned 50/50 split, with and without a bug that drops 10% of the failing users of arm B from the logs
    N = int((120 / (1 - pa)) ** 2)   # N is large enough to see the loss of 10% of the failing users of arm B
    cA = rng.binomial(N, 0.5, S); nB = N - cA; srm = lambda x, y: stats.chi2.sf((x - y) ** 2 / (x + y), 1) < 0.001
    s = rng.binomial(nB, pa); f = rng.binomial(nB - s, 0.9); lift = s / (s + f) - rng.binomial(cA, pa) / cA
    clean_flag, bug_flag = float(srm(cA, nB).mean()), float(srm(cA, s + f).mean()); assert clean_flag < 0.005 and bug_flag > 0.95 and lift.mean() > 0.003
    print(f"E3 sample ratio mismatch (alarm when the chi-square p < 0.001): clean split flagged {clean_flag:.4f}; with the logging bug flagged {bug_flag:.3f}.")
    print(f"   The bug makes a fake lift of {lift.mean():+.3f} although the true lift is 0. A flagged test is not read.")
    # E4. novelty effect: the lift decays from steady + extra to steady; U users per arm per day
    U, days, steady, extra = 1000, np.arange(1, 29), 0.02, 0.08; lt = steady + extra * np.exp(-days / 3)
    ca, cb = rng.binomial(U, pa, (500, 28)), rng.binomial(U, np.clip(pa + lt, 0, 1), (500, 28)); nv = []
    for name, w in (("days 1-3", slice(0, 3)), ("days 15-28", slice(14, 28))):
        k = U * (w.stop - w.start); e = (cb[:, w].sum(1) - ca[:, w].sum(1)) / k; nv.append(dict(window=name, true_lift=lt[w].mean(), estimated=e.mean(), significant=float((zp(ca[:, w].sum(1), k, cb[:, w].sum(1), k) < 0.05).mean())))
    nv = pd.DataFrame(nv); print("E4 novelty effect (true steady lift 0.02):"); print(nv.round(3).to_string(index=False)); nv.round(4).to_csv(f"{OUT_DIR}/prompting_e_novelty.csv", index=False); assert nv.estimated[0] > nv.estimated[1] + 0.03
    print(f"a 3-day test reports {nv.estimated[0]:+.3f}, the steady lift is {steady:+.3f}: it overstates by {nv.estimated[0] - steady:.3f}")
    # E5. guardrail metrics from the measured latency and token numbers: success must rise and neither guardrail may break
    la, lb = D["lat_a"], D["lat_b"]; bs = [np.percentile(rng.choice(lb, len(lb)), 95) / np.percentile(rng.choice(la, len(la)), 95) for _ in range(1000)]
    ca1, cb1 = int(rng.binomial(n, pa)), int(rng.binomial(n, pb)); p = float(zp(ca1, n, cb1, n))
    g = pd.DataFrame([dict(metric="success rate", old=ca1 / n, new=cb1 / n, ratio_or_diff=cb1 / n - ca1 / n, limit="p < 0.05 and > 0", ok=bool(p < 0.05 and cb1 > ca1)),
                      dict(metric="p95 latency s", old=np.percentile(la, 95), new=np.percentile(lb, 95), ratio_or_diff=np.percentile(lb, 95) / np.percentile(la, 95), limit="ratio <= 1.25", ok=bool(np.percentile(lb, 95) / np.percentile(la, 95) <= 1.25)),
                      dict(metric="completion tokens", old=D["ctok_a"], new=D["ctok_b"], ratio_or_diff=D["ctok_b"] / max(D["ctok_a"], 1), limit="ratio <= 1.5", ok=bool(D["ctok_b"] / max(D["ctok_a"], 1) <= 1.5))])
    print(f"E5 guardrails at n={n} per arm (p95 ratio 95% bootstrap interval {np.percentile(bs, 2.5):.2f} to {np.percentile(bs, 97.5):.2f}):"); print(g.round(3).to_string(index=False)); g.round(4).to_csv(f"{OUT_DIR}/prompting_e_guardrails.csv", index=False)
    print("decision:", "SHIP" if g.ok.all() else "DO NOT SHIP, failed: " + ", ".join(g.metric[~g.ok]))
    # E6. Simpson's paradox: B is worse in both segments, but a biased assignment gives B more easy users
    seg = dict(easy=(0.90, 0.85), hard=(0.50, 0.45)); Nn = 20000
    def arm(mix, w): ne = int(rng.binomial(Nn, mix)); ce, ch = int(rng.binomial(ne, seg["easy"][w])), int(rng.binomial(Nn - ne, seg["hard"][w])); return dict(easy=ce / ne, hard=ch / (Nn - ne), overall=(ce + ch) / Nn)
    sp = []
    for name, (ma, mb) in dict(biased=(0.3, 0.8), randomised=(0.55, 0.55)).items():
        x, y = arm(ma, 0), arm(mb, 1); sp.append(dict(assignment=name, easy_old=x["easy"], easy_new=y["easy"], hard_old=x["hard"], hard_new=y["hard"], overall_old=x["overall"], overall_new=y["overall"]))
    sp = pd.DataFrame(sp); r0, r1 = sp.iloc[0], sp.iloc[1]; print("E6 Simpson's paradox:"); print(sp.round(3).to_string(index=False)); sp.round(4).to_csv(f"{OUT_DIR}/prompting_e_simpson.csv", index=False)
    assert abs(0.3 * 0.9 + 0.7 * 0.5 - r0.overall_old) < 0.01 and abs(0.8 * 0.85 + 0.2 * 0.45 - r0.overall_new) < 0.01   # the simulator's expected values
    assert r0.overall_new > r0.overall_old and r0.easy_new < r0.easy_old and r0.hard_new < r0.hard_old and r1.overall_new < r1.overall_old
    print("with the biased split the new prompt wins overall and loses in both segments; with random assignment overall and segments agree")
    # E7. shadow, canary and A/B against a bad new version (true drop of 8 points) and N users in all
    drop, N, d = min(0.08, pa / 2), 20000, max(float((a != b).mean()), 0.2); half = N // 2
    ab = float(((zp(rng.binomial(half, pa, S), half, rng.binomial(half, pa - drop, S), half) < 0.05)).mean())
    looks = np.arange(100, 1001, 100); cb = rng.binomial(100, pa - drop, (S, 10)).cumsum(1); hit = (cb / looks - pa) / np.sqrt(pa * (1 - pa) / looks) < -3
    first = np.where(hit.any(1), hit.argmax(1), 9); exposed = float(looks[first].mean())
    cmp = pd.DataFrame([dict(method="shadow (500 graded pairs)", users_exposed_to_new=0, expected_extra_failures=0, detects_drop=power(500, drop, d), sees_user_behaviour="no"),
                        dict(method="canary 5% (rollback at z<-3)", users_exposed_to_new=exposed, expected_extra_failures=exposed * drop, detects_drop=float(hit.any(1).mean()), sees_user_behaviour="yes, few users"),
                        dict(method="A/B 50/50, fixed horizon", users_exposed_to_new=half, expected_extra_failures=half * drop, detects_drop=ab, sees_user_behaviour="yes, many users")])
    print(f"E7 a bad version (true drop {drop:.2f}), {N} users:"); print(cmp.round(3).to_string(index=False)); cmp.round(3).to_csv(f"{OUT_DIR}/prompting_e_shadow_canary_ab.csv", index=False)
    assert cmp.detects_drop[1] > 0.9 and cmp.detects_drop[2] > 0.99 and cmp.users_exposed_to_new[1] < cmp.users_exposed_to_new[2]
part("E online A/B test simulation", part_e, 300)

## Part F. Business value

Question: how many hours a week does each use case save, and does the ranking change when a failure costs repair time?

The success rates come from part D. The minutes are stated assumptions in the cell, not measurements. Value per task is success times minutes saved, minus failure rate times repair minutes. Break-even success is repair / (saved + repair). Below it, the use case loses time.

In [ ]:
# 9. Part F: business value per use case. Success is measured (or FALLBACK); the minutes are stated assumptions. No API.
MIN_SAVED = dict(triage=2, actions=8, docqa=5, summary=6, sql=15, diagnosis=20)   # assumption: minutes saved when the output is right
PER_WEEK = dict(triage=500, actions=60, docqa=200, summary=80, sql=40, diagnosis=30)   # assumption: tasks per week
REPAIR = [0, 5, 15, 30]                                                             # assumption: minutes to notice and fix one failure
def part_f():
    D = load_inputs(); a, b, uc = D["a"], D["b"], D["uc"]; print("inputs:", D["source"]); print("assumptions, minutes saved per right output:", MIN_SAVED); print("assumptions, tasks per week:", PER_WEEK)
    rows, rank = [], {}
    for r in REPAIR:
        h = {u: PER_WEEK[u] * (b[uc == u].mean() * MIN_SAVED[u] - (1 - b[uc == u].mean()) * r) / 60 for u in UCS}   # hours per week with the new prompt
        rank[r] = pd.Series(h).rank(ascending=False).astype(int)
        rows.append(dict(repair_min=r, **{u: round(v, 1) for u, v in h.items()}, total=round(sum(h.values()), 1)))
    t = pd.DataFrame(rows); print("hours saved per week (new prompt), by repair minutes per failure:"); print(t.to_string(index=False))
    R = pd.DataFrame(rank).T; R.index.name = "repair_min"; print("rank of each use case (1 = most hours):"); print(R.to_string())
    tau = {r: stats.kendalltau(R.loc[0], R.loc[r])[0] for r in REPAIR[1:]}; print("Kendall tau of the ranking against the ranking at repair 0:", {k: round(float(v), 2) for k, v in tau.items()})
    be = pd.DataFrame([dict(use_case=u, success=b[uc == u].mean(), lo=wilson(int(b[uc == u].sum()), 15)[0], hi=wilson(int(b[uc == u].sum()), 15)[1],
                            break_even_at_5=5 / (MIN_SAVED[u] + 5), break_even_at_15=15 / (MIN_SAVED[u] + 15), break_even_at_30=30 / (MIN_SAVED[u] + 30)) for u in UCS])
    be["loses_time_at_15"] = be.success < be.break_even_at_15; print("break-even success = repair / (saved + repair); below it the use case loses time:"); print(be.round(2).to_string(index=False))
    t.to_csv(f"{OUT_DIR}/prompting_f_hours.csv", index=False); R.to_csv(f"{OUT_DIR}/prompting_f_ranks.csv"); be.round(3).to_csv(f"{OUT_DIR}/prompting_f_break_even.csv", index=False)
    top0, top30 = R.loc[0].idxmin(), R.loc[30].idxmin(); worst = [u for u in UCS if be.set_index("use_case").loses_time_at_15[u]]
    print(f"best use case with no repair cost: {top0}; with 30 minutes per failure: {top30}. Use cases that lose time at 15 minutes of repair: {worst or 'none'}")
part("F business value", part_f, 120)

In [ ]:
# 10. Status of every part, a count of calls, and cleanup. This cell holds no server and no GPU job.
pd.DataFrame(ROWS).to_csv(f"{OUT_DIR}/prompting_raw_rows.csv", index=False)
print(pd.DataFrame(STATUS).to_string(index=False))
print(f"API calls used: {CALLS['n']} of {MAX_CALLS} (ok {CALLS['ok']}, failed requests {CALLS['fail']}); total {(time.time() - T0) / 60:.1f} min")
df = pd.DataFrame(ROWS)
for o in (OUTCOMES if len(df) else []):   # why did it fail? up to 3 sample replies (120 characters) for each failure outcome
    for e in df[df.outcome == o].head(3).itertuples(): print(f"[{o}] {e.model} {e.tech} {e.id}: {(e.text or e.err)[:120]!r}")
print("failed attempts by model and status:"); print(pd.DataFrame([dict(model=m, status=c, attempts=n) for (m, c), n in sorted(CODES.items())]).to_string(index=False) if CODES else "none")
print("servers started by this notebook: 0. GPU memory in use by this notebook: 0 MiB (CPU only).")

## What to look for

- **Success columns**: read `success` with `lo` and `hi`. With 15 items per use case, one item moves a rate by 0.07. With 90 items, by 0.011. Compare every rate with the baseline columns.
- **Outcome columns**: `format_error`, `empty`, `cut`, `error` and `not_reached` are failures too. A technique that raises `cut` (chain of thought needs room) or `format_error` pays for it there.
- **McNemar p-value**: it counts only items where the technique and zero-shot differ (`only_zero`, `only_tech`). Compare the gain with the repeat lines: items that changed between two runs show the noise of the API.
- **Pick on dev, measure on test**: only the test numbers count. Forty-two test items are a small sample. Read the Wilson interval.
- **Optimization**: compare the dev gain with the test gain. Four rounds of choice on 48 dev items can pick luck.
- **Offline gate and power**: a gate that fails with a positive gain means the golden set is too small, not that the change is bad. Read the power table for the size you would need.
- **E2**: the false positive rate with 10 looks is far above 0.05. Bonferroni brings it back. **E3**: a flagged test is not read. **E4**: a short test overstates a lift that fades. **E5**: success can rise while a guardrail breaks. **E6**: compare overall rates only when the segment mix is the same.
- **E7**: shadow shows no user behaviour. Canary exposes few users. A/B exposes many users for the full time.
- **Limits**: one API run per cell, small item counts, items built from templates, a simulated online world, and minutes that are assumptions. `FALLBACK` lines are not measurements.

## Exam rules this proves

1. **customization.10**: Prompt engineering changes the input at inference time. The weights do not change (`selecting-large-language-model-customization-techniques.txt`). Zero-shot, few-shot, chain of thought, system prompts and structured output each change success, format errors, tokens and latency. Measure each one on the same items. Do not assume that more examples or more thinking always help.
2. **customization.10**: Structured output asks for a reply that fits a schema. Constrained decoding is meant to guarantee it (`guided-decoding.txt`). Check the `format_error` column to see what the hosted API really did.
3. **customization.10**: Prompt optimization uses errors to write the next prompt. Choose the prompt on dev items. Report held-out items. The gap between dev gain and test gain shows overfitting.
4. **customization.2**: An offline experiment runs old and new on one golden set before release. Pair the items, use an exact paired test, and size the set from the power table.
5. **customization.2**: An A/B test splits users at random. Plan the sample size from the smallest gain worth shipping. Do not peek without a correction. Check the sample ratio first. Read guardrail metrics. Beware of novelty and of mixed segments (Simpson).
6. **customization.2**: Validating model impact before deployment is a chain: offline gate, shadow, canary, A/B test. Each step exposes more users and shows more.
7. **enterprise.5**: A productivity use case pays only when its success is above the break-even. A repair cost per failure can change which use case ranks first.

| Exam term | One plain line | Shown in |
|---|---|---|
| Prompt engineering | Change the input text, not the weights. | Parts B and C |
| System prompts | Standing rules before the task. | B (`system_rules`), C |
| Zero-shot prompting | The task with no example. | A, B |
| Few-shot prompting | The task with a few solved examples before it. | B (`few_1`, `few_3`, `few_5`) |
| Chain-of-thought prompting | Ask for step-by-step thinking, then the answer. | B (`cot`) |
| Structured output | A reply that fits a JSON schema. | B (`structured`) |
| Prompt optimization | A model rewrites the prompt from the errors. | C |
| Offline experiments | Old and new on one fixed set, before release. | D |
| A/B testing | Users split at random between old and new. | E |
| Validating model impact before deployment | Offline gate, shadow, canary and A/B. | D, E7 |
| Productivity use cases for LLM-powered applications | Office tasks where a model saves time. | A, F |